In [2]:
homedir = '/u/az6922/DRing/src/emp/datacentre/'
import random
import numpy as np

# from makec2s.ipynb
def genflowbytes():
    np.random.seed(0)
    
    mean_bytes = 100.0 * 1024
    shape = 1.05
    scale = mean_bytes * (shape - 1)/shape

    x = np.random.exponential(scale=1.0/shape)
    flowbytes = int(scale * np.exp(x))
    return flowbytes

def adjustbytesbymtu(flowbytes):
  mss = 1500
  return mss * ((flowbytes+mss-1)//mss)

large_flow_threshold = 10 * 1024 * 1024

In [3]:
stime = 144 # ms
nlinks = 2048
nhosts = 3072
bw = 1342176000 # B per second
load_list = range(1,10) #[10,30,50,70]
seed_list = [1,2,3,4,5]
topologytype = 1
nswitches = 80
os = 1
k = 64
nintervals = 1
npfile = 'evalnetpathfiles/netpath_leafspine_80_64_ecmp.np'
pwfile = 'experiments/nsdi26fall/eval_main/unv1/pwfiles/pathweight_leafspine.pw'

(current dir: ~/DRing/src/emp/datacentre/experiments/nsdi26fall/)
cp test_general_setup/pwfiles/pathweight_leafspine.pw eval_main/unv1/pwfiles/pathweight_leafspine.pw

In [4]:
# generate connection_matrices file (1)
unv1bytes = 0
unv1file = f'{homedir}rawtrafficfiles/unv1'
maxinterval = 0
with open(unv1file, 'r') as f:
    lines = f.readlines()
    for line in lines:
        tokens = line.split(',')
        # 0,32,31,10500
        # interval,fromserver,toserver,bytes
        unv1bytes += int(tokens[3])
        maxinterval = max(maxinterval, int(tokens[0]))
print(f'unv1bytes {unv1bytes}, maxinterval {maxinterval}, fullload {bw * stime * nlinks / 1000}, ratio {(bw * stime * nlinks / 1000) / unv1bytes}')

unv1bytes 162036861000, maxinterval 7, fullload 395823808512.0, ratio 2.4428010149616513


In [5]:
# generate connection_matrices file (2)
random.seed(0)
for load in load_list:
    totalbytes = bw * stime / 1000 * nlinks * load / 100  # B
    mult = totalbytes / unv1bytes
    actualbytes = 0
    cmfile = f'cmfiles/leafspine_load{load}.cm'
    with open(cmfile, 'w') as fw:
        with open(unv1file, 'r') as fr:
            lines = fr.readlines()
            iline = 0
            while actualbytes < totalbytes:
                line = lines[iline]
                tokens = line.split(',')
                interval = int(tokens[0])
                fromserver = int(tokens[1])
                toserver = int(tokens[2])
                multbytes = int(tokens[3])

                if fromserver >= nhosts or toserver >= nhosts:
                    iline += 1
                    if iline >= len(lines):
                        iline = 0
                        if mult-1>0:
                            mult = mult-1
                    continue

                if mult >= 1 or (random.random() < mult):
                    multbytes = adjustbytesbymtu(multbytes)
    
                    # generate random start time
                    start_time_ms = random.uniform(0, stime//(maxinterval+1)) + interval * (stime//(maxinterval+1))

                    fw.write(f'{fromserver},{toserver},{int(multbytes)},{start_time_ms:.4f}\n')
                    actualbytes += int(multbytes)

                iline += 1
                if iline >= len(lines):
                    iline = 0
                    if mult-1>0:
                        mult = mult-1

                    # print(f'actualbytes {actualbytes}, totalbytes {totalbytes}, mult {mult}', end='\r')

    print(f'load {load}%, totalbytes {totalbytes}, unv1bytes {unv1bytes}, mult {mult}, actualbytes {actualbytes}')


load 1%, totalbytes 3958238085.12, unv1bytes 162036861000, mult 0.02442801014961651, actualbytes 3958267500
load 2%, totalbytes 7916476170.24, unv1bytes 162036861000, mult 0.04885602029923302, actualbytes 7916959500
load 3%, totalbytes 11874714255.36, unv1bytes 162036861000, mult 0.07328403044884954, actualbytes 11875021500
load 4%, totalbytes 15832952340.48, unv1bytes 162036861000, mult 0.09771204059846604, actualbytes 15832962000
load 5%, totalbytes 19791190425.6, unv1bytes 162036861000, mult 0.12214005074808255, actualbytes 19791237000
load 6%, totalbytes 23749428510.72, unv1bytes 162036861000, mult 0.14656806089769908, actualbytes 23750587500
load 7%, totalbytes 27707666595.84, unv1bytes 162036861000, mult 0.1709960710473156, actualbytes 27707680500
load 8%, totalbytes 31665904680.96, unv1bytes 162036861000, mult 0.19542408119693208, actualbytes 31666719000
load 9%, totalbytes 35624142766.08, unv1bytes 162036861000, mult 0.21985209134654862, actualbytes 35624275500


In [6]:
# generate conf file
conffile = f'{homedir}experiments/nsdi26fall/eval_main/unv1/run_ls.conf'
with open(conffile, 'w') as f:
    for seed in seed_list:
        for load in load_list:
            cmfile = f'experiments/nsdi26fall/eval_main/unv1/cmfiles/leafspine_load{load}.cm'
            outfile = f'experiments/nsdi26fall/eval_main/unv1/outfiles/leafspine_load{load}_seed{seed}.out'
            f.write(f"./eval -stime {stime} -seed {seed} -cmfile {cmfile} -topologytype {topologytype} -numswitches {nswitches} -numhosts {nhosts} -os {os} -ls_k {k} -npfile {npfile} -pwfileprefix {pwfile} -numintervals {nintervals} > {outfile}\n")
            

python3 pararun.py --conf experiments/nsdi26fall/eval_main/unv1/run_ls.conf --worker 45